# Day 3 Exploratory Data Analysis
## 1. Project Overview
This notebook covers the Exploratory Data Analysis for the Bluestock Fintech Mutual Fund Analytics Platform Capstone.
We analyze various aspects including NAV trends, AUM growth, SIP inflows, and investor demographics.


## 2. Data Loading
Loading the processed datasets for analysis.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import os

# Ensure charts directory exists
os.makedirs('../reports/charts', exist_ok=True)

# Load datasets
nav_df = pd.read_csv('../data/processed/clean_nav_history.csv')
aum_df = pd.read_csv('../data/processed/clean_aum_by_fund_house.csv')
sip_df = pd.read_csv('../data/processed/clean_monthly_sip_inflows.csv')
category_df = pd.read_csv('../data/processed/clean_category_inflows.csv')
investor_df = pd.read_csv('../data/processed/clean_investor_transactions.csv')
folio_df = pd.read_csv('../data/processed/clean_industry_folio_count.csv')
holdings_df = pd.read_csv('../data/processed/clean_portfolio_holdings.csv')
master_df = pd.read_csv('../data/processed/clean_fund_master.csv')
benchmark_df = pd.read_csv('../data/processed/clean_benchmark_indices.csv')

# Format dates
nav_df['date'] = pd.to_datetime(nav_df['date'])
aum_df['date'] = pd.to_datetime(aum_df['date'])
sip_df['month'] = pd.to_datetime(sip_df['month'])
investor_df['transaction_date'] = pd.to_datetime(investor_df['transaction_date'])
folio_df['month'] = pd.to_datetime(folio_df['month'])
benchmark_df['date'] = pd.to_datetime(benchmark_df['date'])


## 3. Data Quality Verification
Quick check of dataset shapes and missing values.

In [2]:
print(f"NAV Data Shape: {nav_df.shape}")
print(f"Investor Data Shape: {investor_df.shape}")


NAV Data Shape: (46000, 3)
Investor Data Shape: (32778, 13)


## 4. NAV Analysis
### Chart 1-3: NAV Trend Analysis
Visualizing daily NAV for major schemes. Highlighting 2023 bull run and 2024 market correction.

In [3]:

# Filter data
nav_plot_df = nav_df[(nav_df['date'] >= '2022-01-01') & (nav_df['date'] <= '2026-05-31')]

# Merge with master to get scheme names
nav_master = pd.merge(nav_plot_df, master_df[['amfi_code', 'scheme_name']], on='amfi_code')

# Chart 1: All schemes
fig = px.line(nav_master, x='date', y='nav', color='scheme_name', 
              title='Daily NAV for Mutual Fund Schemes (Jan 2022 - May 2026)')

fig.update_layout(
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    )
)
# Add annotations for bull run and market correction
fig.add_vrect(x0="2023-01-01", x1="2023-12-31", fillcolor="green", opacity=0.1, line_width=0, annotation_text="2023 Bull Run")
fig.add_vrect(x0="2024-06-01", x1="2024-12-31", fillcolor="red", opacity=0.1, line_width=0, annotation_text="2024 Market Correction")

fig.write_image("../reports/charts/01_nav_trend_all_schemes.png", width=1200, height=800)

# Chart 2: Top 5 performing schemes NAV
top_schemes = nav_master.groupby('scheme_name')['nav'].last().nlargest(5).index
fig2 = px.line(nav_master[nav_master['scheme_name'].isin(top_schemes)], x='date', y='nav', color='scheme_name',
               title='Daily NAV - Top 5 Schemes')
fig2.write_image("../reports/charts/02_nav_trend_top5.png", width=1000, height=600)

# Chart 3: Bottom 5 schemes NAV
bottom_schemes = nav_master.groupby('scheme_name')['nav'].last().nsmallest(5).index
fig3 = px.line(nav_master[nav_master['scheme_name'].isin(bottom_schemes)], x='date', y='nav', color='scheme_name',
               title='Daily NAV - Bottom 5 Schemes')
fig3.write_image("../reports/charts/03_nav_trend_bottom5.png", width=1000, height=600)


## 5. AUM Analysis
### Chart 4: AUM Growth Analysis
Analyzing Asset Under Management growth across fund houses.

In [4]:

aum_df['year'] = aum_df['date'].dt.year
aum_yearly = aum_df.groupby(['year', 'fund_house'])['aum_crore'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=aum_yearly, x='year', y='aum_crore', hue='fund_house')
plt.title('AUM Growth Analysis by Fund House')
plt.ylabel('AUM (Crore)')
plt.xlabel('Year')
# Highlight SBI Mutual Fund leadership
plt.axhline(y=1250000, color='r', linestyle='--', label='₹12.5 Lakh Crore (SBI Peak)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../reports/charts/04_aum_growth_analysis.png')
plt.close()


## 6. SIP Analysis
### Chart 5: SIP Inflow Trend
Visualizing the monthly SIP inflow trend.

In [5]:

sip_plot = sip_df[(sip_df['month'] >= '2022-01-01') & (sip_df['month'] <= '2025-12-31')].copy()
fig = px.line(sip_plot, x='month', y='sip_inflow_crore', markers=True, title='Monthly SIP Inflows (Jan 2022 - Dec 2025)')

# Add trend line using numpy polyfit to avoid statsmodels dependency
import numpy as np
x_numeric = pd.to_numeric(sip_plot['month'])
z = np.polyfit(x_numeric, sip_plot['sip_inflow_crore'], 1)
p = np.poly1d(z)
fig.add_scatter(x=sip_plot['month'], y=p(x_numeric), mode='lines', name='Trend Line', line=dict(dash='dash', color='red'))

fig.add_annotation(x="2025-12-01", y=31002, text="₹31,002 Cr", showarrow=True, arrowhead=1, ax=-40, ay=-40, 
                   bordercolor="black", borderwidth=1, borderpad=4, bgcolor="white", opacity=0.8)

fig.write_image("../reports/charts/05_sip_inflow_trend.png", width=1000, height=600)


## 7. Category Analysis
### Chart 6: Category Inflow Heatmap
Net inflows by fund category over time.

In [6]:

category_pivot = category_df.pivot_table(index='category', columns='month', values='net_inflow_crore', aggfunc='sum')
plt.figure(figsize=(14, 8))
sns.heatmap(category_pivot, cmap='RdYlGn', annot=False)
plt.title('Net Inflow by Fund Category (Crore)')
plt.tight_layout()
plt.savefig('../reports/charts/06_category_inflow_heatmap.png')
plt.close()


## 8. Investor Analysis
### Chart 7-9: Investor Demographics
Analyzing age, SIP amounts, and gender distribution.

In [7]:

# Chart 7: Age group distribution pie chart
age_counts = investor_df['age_group'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Age Group Distribution')
plt.savefig('../reports/charts/07_age_group_distribution.png')
plt.close()

# Chart 8: SIP amount box plot by age group
plt.figure(figsize=(10, 6))
sns.boxplot(data=investor_df[investor_df['transaction_type'] == 'SIP'], x='age_group', y='amount_inr', order=['18-25', '26-35', '36-45', '46-55', '56-60', '60+'])
plt.title('SIP Amount Distribution by Age Group')
plt.yscale('log')
plt.savefig('../reports/charts/08_sip_amount_by_age.png')
plt.close()

# Chart 9: Gender split chart
gender_counts = investor_df['gender'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
plt.title('Gender Split')
plt.savefig('../reports/charts/09_gender_split.png')
plt.close()


## 9. Geographic Analysis
### Chart 10-11: Geographic Distribution
Analyzing contributions by state and city tiers.

In [8]:

# Chart 10: Horizontal bar chart Total SIP amount by state
sip_transactions = investor_df[investor_df['transaction_type'] == 'SIP']
state_sip = sip_transactions.groupby('state')['amount_inr'].sum().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
state_sip.plot(kind='barh', color='teal')
plt.title('Total SIP Amount by State')
plt.xlabel('Amount (INR)')
plt.tight_layout()
plt.savefig('../reports/charts/10_sip_by_state.png')
plt.close()

# Chart 11: Pie chart T30 vs B30 city tier
tier_counts = investor_df.groupby('city_tier')['amount_inr'].sum()
plt.figure(figsize=(8, 8))
plt.pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', colors=['lightgreen', 'lightblue'])
plt.title('Investment Amount: T30 vs B30 Cities')
plt.savefig('../reports/charts/11_city_tier_distribution.png')
plt.close()


## 10. Folio Analysis
### Chart 12: Folio Count Growth
Growth in industry folios.

In [9]:

plt.figure(figsize=(12, 6))
plt.plot(folio_df['month'], folio_df['total_folios_crore'], marker='o', linestyle='-', color='purple')
plt.title('Industry Folio Count Growth')
plt.ylabel('Total Folios (Crore)')
plt.xlabel('Month')

# Annotations
start_val = folio_df['total_folios_crore'].iloc[0]
end_val = folio_df['total_folios_crore'].iloc[-1]
plt.annotate(f"{start_val} Cr", xy=(folio_df['month'].iloc[0], start_val), xytext=(10, 10), textcoords='offset points')
plt.annotate(f"{end_val} Cr", xy=(folio_df['month'].iloc[-1], end_val), xytext=(-30, 10), textcoords='offset points')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/charts/12_folio_count_growth.png')
plt.close()


## 11. Correlation Analysis
### Chart 13: NAV Return Correlation Matrix
Daily returns correlation for top 10 schemes.

In [10]:

# Calculate daily returns for top 10 schemes by latest NAV
top_10_codes = nav_df.groupby('amfi_code')['nav'].last().nlargest(10).index
nav_top_10 = nav_df[nav_df['amfi_code'].isin(top_10_codes)]

# Pivot to get date on index, amfi_code on columns
nav_pivot = nav_top_10.pivot(index='date', columns='amfi_code', values='nav')
daily_returns = nav_pivot.pct_change().dropna()

# Map AMFI code to Scheme Name for better readability
scheme_map = dict(zip(master_df['amfi_code'], master_df['scheme_name'].str[:20]))
daily_returns.rename(columns=scheme_map, inplace=True)

corr_matrix = daily_returns.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Daily Returns Correlation Matrix - Top 10 Schemes')
plt.tight_layout()
plt.savefig('../reports/charts/13_nav_correlation_matrix.png')
plt.close()


## 12. Portfolio Analysis
### Chart 14: Sector Allocation Donut
Sector weights across all equity funds.

In [11]:

sector_weights = holdings_df.groupby('sector')['weight_pct'].mean().sort_values(ascending=False).head(10)
other_weight = holdings_df.groupby('sector')['weight_pct'].mean().sort_values(ascending=False).iloc[10:].sum()
sector_weights['Others'] = other_weight

plt.figure(figsize=(10, 10))
plt.pie(sector_weights, labels=sector_weights.index, autopct='%1.1f%%', startangle=90, wedgeprops=dict(width=0.4))
plt.title('Top 10 Sector Allocation (Donut Chart)')
plt.savefig('../reports/charts/14_sector_allocation_donut.png')
plt.close()


## 13. Additional Visuals
### Chart 15+: Extra Analysis
Exploring risk categories, fund categories, and benchmark index trends.

In [12]:

# Chart 15: Risk Category Distribution
plt.figure(figsize=(8, 6))
sns.countplot(data=master_df, y='risk_category', palette='viridis')
plt.title('Distribution of Schemes by Risk Category')
plt.tight_layout()
plt.savefig('../reports/charts/15_risk_category_dist.png')
plt.close()

# Chart 16: Fund Category Distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=master_df, y='category', palette='Set2')
plt.title('Distribution of Schemes by Fund Category')
plt.tight_layout()
plt.savefig('../reports/charts/16_fund_category_dist.png')
plt.close()

# Chart 17: Benchmark Index Trend Overview
plt.figure(figsize=(12, 6))
sns.lineplot(data=benchmark_df, x='date', y='close_value', hue='index_name')
plt.title('Benchmark Index Trend Overview')
plt.yscale('log')
plt.tight_layout()
plt.savefig('../reports/charts/17_benchmark_index_trend.png')
plt.close()


C:\Users\danyb\AppData\Local\Temp\ipykernel_13020\3686974098.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=master_df, y='risk_category', palette='viridis')
C:\Users\danyb\AppData\Local\Temp\ipykernel_13020\3686974098.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=master_df, y='category', palette='Set2')


## 14. Key Findings

Here are the 10 key findings from our Exploratory Data Analysis:

**Insight 1:**  
**Observation:** SBI Mutual Fund maintains the highest AUM across all years, distinctly leading the market.  
**Supporting chart reference:** Chart 4: AUM Growth Analysis by Fund House

**Insight 2:**  
**Observation:** SIP inflows reached an all-time high of ₹31,002 Cr in Dec 2025, showing a strong upward trend since 2022.  
**Supporting chart reference:** Chart 5: SIP Inflow Trend

**Insight 3:**  
**Observation:** The NAV trends show significant volatility, but almost all schemes experienced a major recovery during the 2023 Bull Run.  
**Supporting chart reference:** Chart 1: Daily NAV for Mutual Fund Schemes

**Insight 4:**  
**Observation:** Maharashtra contributes the highest transaction volume/amount compared to other states.  
**Supporting chart reference:** Chart 10: Total SIP Amount by State

**Insight 5:**  
**Observation:** T30 cities still dominate the investment amount, but B30 cities account for a substantial and growing portion.  
**Supporting chart reference:** Chart 11: Investment Amount: T30 vs B30 Cities

**Insight 6:**  
**Observation:** Total industry folios have nearly doubled, growing steadily towards 26.12 Cr.  
**Supporting chart reference:** Chart 12: Industry Folio Count Growth

**Insight 7:**  
**Observation:** Mid-cap and Small-cap categories attracted stronger inflows during the analyzed period.  
**Supporting chart reference:** Chart 6: Category Inflow Heatmap

**Insight 8:**  
**Observation:** The majority of SIP investors fall within the 25-45 age group, with males being the primary investors.  
**Supporting chart reference:** Chart 7 & 9: Age Group Distribution and Gender Split

**Insight 9:**  
**Observation:** Financial Services and Information Technology represent the most dominant sector allocations across equity funds.  
**Supporting chart reference:** Chart 14: Top 10 Sector Allocation (Donut Chart)

**Insight 10:**  
**Observation:** There is a very strong positive correlation (>0.9) among the daily returns of the top-performing equity schemes.  
**Supporting chart reference:** Chart 13: Daily Returns Correlation Matrix - Top 10 Schemes
